In [ ]:
# pip install polars entsoe-py sqlite3

In [ ]:
'''
About Polars 

1. Parallel Execution: Polars is written in Rust and utilizes 
all your CPU cores natively. Pandas typically runs on a single core.
2. Fast SQL Ingestion: Writing a Polars DataFrame to SQLite 
using .write_database() is highly optimized compared to Pandas' .to_sql().
3.0Blazing Fast Feature Engineering: When you start creating Lag features (e.g., shift(24))
or rolling averages across millions of rows of energy data, 
Polars will execute them in milliseconds.

Utilized Polars paired with ADBC (Arrow Database Connectivity) drivers for optimized,
low-latency data streams directly into the SQL layer
'''

In [10]:
'''
Pull Day-ahead data from ENTSO-E in Nordic bidding zones into the Database
The script combines pulling data using API key into Pandas Dataframe,
converting into Polars Dataframe for faster query,
and writing database with SQLite3
'''

from pathlib import Path
import pandas as pd
import polars as pl
from entsoe import EntsoePandasClient
import sqlite3
# pip install adbc-driver-sqlite

# 1. API Token from ENTSO-E
cwd = Path.cwd()
api_file = cwd.parent.parent / "entsoe-api.txt"
API_TOKEN = api_file.read_text(encoding="utf-8").strip()
client = EntsoePandasClient(api_key=API_TOKEN)

# 2. start and end time for 1 week (hottest days in Sweden between 28-29 June 2026)
start = pd.Timestamp('2026-06-14', tz='Europe/Berlin')
end = pd.Timestamp('2026-06-29', tz='Europe/Berlin')

# 3. Bidding Zone 
nordic_zones = [
    'DK_1', 'DK_2',                                     # Denmark
    'SE_1', 'SE_2', 'SE_3', 'SE_4',                     # Sweden
    'NO_1', 'NO_2', 'NO_3', 'NO_4', 'NO_5'              # Norway
]
# 4. Prepare database
db_uri = "sqlite://nordic_energy_market.db" # Polars uses connection URIs
connection = sqlite3.connect('nordic_energy_market.db') # Polars doesn't need cursor to interact with the database
connection.execute('''
    CREATE TABLE IF NOT EXISTS master_day_ahead_prices (
        timestamp TEXT,
        bidding_zone TEXT,
        price_eur_mwh REAL,
        PRIMARY KEY (timestamp, bidding_zone)
    )
''')
connection.close()


# 5. Pull Day-Ahead Prices for 1 day
try:
    for country_code in nordic_zones:
        print(f"Fetching data via API for: {country_code}...")
        df_prices = client.query_day_ahead_prices(country_code, start=start, end=end)
    
    # Formatting DataFrame for SQL
        df_prices = df_prices.reset_index()
        df_prices.columns = ['timestamp', 'price_eur_mwh']
        
    # Convert dataframe from pandas to polars
        df = pl.from_pandas(df_prices)
    
    # Convert datetime to String avoid issue in SQL, add bidding zone column, and rearrange columns
        df = df.with_columns([
            pl.col('timestamp').dt.strftime('%Y-%m-%d %H:%M:%S'),
            pl.lit(country_code).alias('bidding_zone')])
        df = df.select(['timestamp', 'bidding_zone', 'price_eur_mwh'])
            
        df.write_database(
            table_name="master_day_ahead_prices",
            connection=db_uri,
            if_table_exists="append",
            engine="adbc" # Auto-chooses the fastest driver available
        )
        print(f"Polars successfully wrote {len(df)} rows for {country_code}.")
        
except Exception as e:
    print(f"Error fetching data: {e}")


Fetching data via API for: DK_1...
Polars successfully wrote 1441 rows for DK_1.
Fetching data via API for: DK_2...
Polars successfully wrote 1441 rows for DK_2.
Fetching data via API for: SE_1...
Polars successfully wrote 1441 rows for SE_1.
Fetching data via API for: SE_2...
Polars successfully wrote 1441 rows for SE_2.
Fetching data via API for: SE_3...
Polars successfully wrote 1441 rows for SE_3.
Fetching data via API for: SE_4...
Polars successfully wrote 1441 rows for SE_4.
Fetching data via API for: NO_1...
Polars successfully wrote 1441 rows for NO_1.
Fetching data via API for: NO_2...
Polars successfully wrote 1441 rows for NO_2.
Fetching data via API for: NO_3...
Polars successfully wrote 1441 rows for NO_3.
Fetching data via API for: NO_4...
Polars successfully wrote 1441 rows for NO_4.
Fetching data via API for: NO_5...
Polars successfully wrote 1441 rows for NO_5.
